In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
words = open("names.txt", "r").read().splitlines()

chars = sorted(list(set("".join(words))))

stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0

itos = {i: ch for ch, i in stoi.items()}

VOCAB_SIZE = len(stoi)



xs = []
ys = []

for word in words:
    
    chs = ["."] + list(word) + ["."]
    
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

num = xs.nelement()

print("training pairs:", num)

In [ ]:
g = torch.Generator().manual_seed(2147483647)

W = torch.randn((VOCAB_SIZE, VOCAB_SIZE), generator=g, requires_grad=True)

In [ ]:
for step in range(100):

    logits = W[xs]                 # row lookup

    loss = F.cross_entropy(logits, ys)

    # L2 regularization
    loss = loss + 0.01 * (W ** 2).mean()

    W.grad = None
    loss.backward()

    W.data += -50 * W.grad

    if step % 10 == 0:
        print(f"step {step:3d} | loss {loss.item():.4f}")
